# 02 · Slang: the vibe is not in the words people search with

Notebook 01 showed the vibe is in the reviews. Here we show **keyword search can't reach it**:
someone searches "boujee", but reviewers write "fancy", "upscale" or "classy".

1. **Gap table** (the headline): for slang query words, how many reviews and places use the slang
   vs. its plain meaning. The words were picked from our example queries *before* counting, and
   every row is shown, including the ones where slang does appear.
2. **Urban Dictionary lexicon**: the slang list our query expansion will use later.
3. **Which slang reviewers actually use**: spaCy sentences, the share containing slang, and the top 30 terms.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from tqdm.auto import tqdm

from vibecheck.filter_sf import PROCESSED_DIR, REVIEWS_OUT
from vibecheck.lexicon import UD_CSV, build_slang_lexicon, find_terms, gap_table, load_ud_definitions
from vibecheck.plots import MUTED, SINGLE_COLOR, use_style
from vibecheck.report import write_section

use_style()
reviews = pd.read_parquet(REVIEWS_OUT)
print(f"{len(reviews):,} reviews, {reviews['gmap_id'].nunique():,} places")

## 1. The gap table

In [ ]:
gap = gap_table(reviews)
gap

In [ ]:
# Places keyword search can reach with the slang word vs. with its plain meaning (log scale).
SLANG_COLOR, PLAIN_COLOR = "#eb6834", "#2a78d6"  # validated categorical slots 2 and 1
order = gap.sort_values("places (plain)")
y = np.arange(len(order))
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(y + 0.2, order["places (plain)"].clip(lower=0.8), height=0.38, color=PLAIN_COLOR, label="plain meaning")
ax.barh(y - 0.2, order["places (slang)"].clip(lower=0.8), height=0.38, color=SLANG_COLOR, label="slang word")
for i, (s, p) in enumerate(zip(order["places (slang)"], order["places (plain)"], strict=True)):
    ax.text(max(s, 0.8) * 1.1, i - 0.2, f"{s:,}", va="center", color=MUTED, fontsize=8)
    ax.text(p * 1.1, i + 0.2, f"{p:,}", va="center", color=MUTED, fontsize=8)
ax.set_yticks(y, order["query"])
ax.set_xscale("log")
ax.set(title="Places a keyword search can find: slang vs. plain meaning", xlabel="places (log scale)")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
fig.savefig("../reports/gap_chart.png")
plt.show()

## 2. Urban Dictionary slang lexicon

Rules (constants at the top of `lexicon.py`):

- a definition with at least 100 upvotes and an upvote ratio of at least 0.7
- single words: rare in normal English (wordfreq Zipf < 3) but seen somewhere (Zipf > 0)
- two-word phrases like "low key": at least 1,000 upvotes (Zipf can't judge phrases)
- drop terms whose definitions look offensive or describe a person's name

Note: the dump's votes stop in 2016, so newer slang ("rizz", "cheugy") isn't in it.

In [ ]:
definitions = load_ud_definitions(UD_CSV)
lexicon = build_slang_lexicon(definitions)
lexicon.to_parquet(PROCESSED_DIR / "slang_lexicon.parquet", index=False)
print(f"{len(definitions):,} well-voted definitions -> {len(lexicon):,} slang terms "
      f"({lexicon['term'].str.contains(' ').sum():,} two-word phrases)")
lexicon.head(15)

## 3. Which slang do reviewers actually use?

We split reviews into sentences with spaCy (only the fast sentence segmenter is enabled) and look up
every word and two-word phrase in the lexicon.

Read the share with care: a high number doesn't mean keyword search works (see the gap table),
and a low one doesn't mean slang doesn't matter. Users type slang even when reviewers don't.
The useful part is *which* terms show up.

In [ ]:
nlp = spacy.load("en_core_web_sm", exclude=["parser", "ner", "lemmatizer", "tagger", "attribute_ruler"])
nlp.enable_pipe("senter")
terms = set(lexicon["term"])

n_sentences, n_with_slang, term_counts = 0, 0, Counter()
for doc in tqdm(nlp.pipe(reviews["text"], batch_size=512), total=len(reviews), desc="sentences"):
    for sent in doc.sents:
        found = find_terms(sent.text, terms)
        n_sentences += 1
        n_with_slang += bool(found)
        term_counts.update(set(found))

slang_share = n_with_slang / n_sentences
print(f"{n_sentences:,} sentences, {slang_share:.1%} contain at least one Urban Dictionary slang term")

In [ ]:
top30 = pd.DataFrame(term_counts.most_common(30), columns=["term", "sentences"])
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(top30["term"][::-1], top30["sentences"][::-1], color=SINGLE_COLOR, height=0.7)
ax.bar_label(ax.containers[0], fmt="{:,.0f}", padding=3, color=MUTED, fontsize=8)
ax.set(title="30 most common slang terms in SF reviews", xlabel="sentences")
ax.grid(axis="y", visible=False)
fig.savefig("../reports/top_slang.png")
plt.show()

## 4. Write to `reports/numbers.md`

In [ ]:
md = f"""## Slang: the vibe is not in the words people search with (notebook 02)

Gap table. Query words were chosen from our example queries before counting; all rows shown.

{gap.to_markdown(index=False, floatfmt=",.0f")}

- Urban Dictionary lexicon: {len(lexicon):,} slang terms ({lexicon['term'].str.contains(' ').sum():,} two-word phrases), from {len(definitions):,} well-voted definitions.
- {n_sentences:,} review sentences (spaCy); {slang_share:.1%} contain at least one lexicon term.

30 most common slang terms (sentences containing them):
{", ".join(f"{t} ({n:,})" for t, n in top30.itertuples(index=False))}
"""
write_section("slang", md)
print(md)